# 04 — Confidence calibration and binomial-bound selective evaluation (v2)
**Input: Notebooks 01–03 v2. Run from the same working directory after Notebook 03 has completed all six model/seed runs.** You need the full v2 model files and the v2 prepared image cache. CPU inference is used; this notebook does not retrain or download images.

The fixed main experiment is unchanged: fit one positive temperature per checkpoint using the **calibration cameras**, then independently evaluate the two models and three paired seeds on the **test cameras**. The secondary selective-classification rule is amended before final test evaluation: select thresholds using the **one-sided Clopper–Pearson binomial upper bound with the existing Bonferroni correction**. Observed error alone is not the selection criterion.

Keep the predefined grid, 5% target, at least 100 accepted policy images and at least five represented policy cameras. A zero-error threshold may be selected when its binomial bound passes. The approximate simultaneous camera-bootstrap analysis remains a **diagnostic only**: a high, unavailable or degenerate camera estimate does not veto selection. Zero bootstrap variation is never interpreted as zero uncertainty.

The binomial confidence interpretation assumes independent accepted events; within-camera dependence may violate that assumption. A selected threshold is not a camera-aware certificate or a guarantee for a new camera. All threshold calculations and camera diagnostics use **policy cameras**, not test outcomes.

Workflow: verify v2 provenance → fit temperatures on calibration → screen policy thresholds using the binomial bound and report camera diagnostics → freeze temperatures/thresholds → infer held-out test scores → report test outcomes, uncertainty and figures.

**Test-set firewall:** Never use the test data to revise the threshold grid, uncertainty method, target error, selection rule, model, or temperature. Existing frozen evaluation/decision files from a different method are deliberately rejected; never overwrite them or re-baseline their cached prediction hashes to adopt this amendment.


## 1. Freeze the v2 calibration, policy-screening and comparison methods
Temperature fitting remains unchanged: minimize event-weighted calibration NLL over positive
T in [0.01, 100]. Calibration cameras fit T; policy cameras select thresholds; test cameras
only evaluate the frozen procedure. The predefined confidence grid, Bonferroni family (all
12 policies and all 85 cutoffs), 5% error target, minimum support, and bootstrap settings remain unchanged.

**Threshold selection:** require the one-sided Clopper–Pearson bound with Bonferroni adjustment
<=5%, at least 100 accepted images and at least five accepting policy cameras. Select the greatest
accepted count, breaking ties with the higher cutoff. Zero observed errors are allowed when this
bound passes. The binomial calculation has nominal familywise 95% coverage under independently
sampled accepted-event assumptions. Within-camera dependence may violate those assumptions;
this is not a 95% camera-aware guarantee or certification.

**Camera diagnostic only:** retain the shared whole-camera bootstrap and its approximate
simultaneous standardized-bootstrap upper estimates. Zero observed errors, near-zero bootstrap
standard deviation or nonfinite estimates are labelled unavailable camera uncertainty. Empty
bootstrap acceptance retains the original conservative 100%-risk assignment, but does not rescue
unidentified zero-error uncertainty. Unavailable estimates stay missing, never zero. Neither
availability nor the size of the camera estimate affects threshold selection. A passing binomial
bound must not be interpreted as establishing low error at a new camera.

**Primary comparison:** NLL calibration gain = raw NLL minus scaled NLL. Compare EfficientNet's
gain minus MobileNet's gain, averaging the three paired seeds within each shared test-camera
bootstrap draw. A positive difference favours a larger EfficientNet improvement; a negative
one favours MobileNet. A zero-crossing interval does not establish equivalence. This comparison
is specified before test inference. Existing Brier/ECE and other contrasts remain supporting analyses.

Main test metrics remain event-weighted; the full camera audit belongs in Notebook 05 v2.
No manual image review is performed in this experiment; abstention means no automatic acceptance.
No test results may be used to revise the grid, uncertainty method, training, temperature fitting,
data partitions, or primary comparison.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, io, json, platform, sys, time
import numpy as np
import pandas as pd
from scipy.stats import beta as beta_distribution
import scipy
from PIL import Image
import PIL
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
from IPython.display import display

# Same project root and experiment ID as Notebook 03 v2.
PROJECT_ROOT = None  # Example: Path.home() / 'ml_project' / 'wildlife_confidence_project_v2'
EXPERIMENT_ID = 'five_categories_camera_balanced_v2'
TRAIN_ID = 'training_v2_corrected'
if PROJECT_ROOT is None:
    possible = [Path.cwd() / 'wildlife_confidence_project_v2', Path.cwd()]
    candidates = list(dict.fromkeys(p.resolve() for p in possible if (
        p / 'reports' / EXPERIMENT_ID / TRAIN_ID / 'training_index.json').is_file()))
    if len(candidates) != 1:
        raise FileNotFoundError('Cannot locate Notebook 03 v2 training_index.json. '
                                'Run it first or set PROJECT_ROOT to the existing v2 project.')
    ROOT = candidates[0]
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()

PREPARED = ROOT / 'data/prepared'
EXP = PREPARED / EXPERIMENT_ID
PREP_REPORT = ROOT / 'reports' / EXPERIMENT_ID
TRAIN_REPORT = PREP_REPORT / TRAIN_ID
REPORT = PREP_REPORT / 'evaluation_v2'
CACHE = ROOT / 'models' / EXPERIMENT_ID / 'evaluation_v2'
for directory in (REPORT, CACHE):
    directory.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cpu')
torch.set_num_threads(min(4, max(1, torch.get_num_threads())))

# POLICY_SETTINGS is part of the frozen evaluation protocol. Do not change after viewing test outcomes.
# The grid is fixed BEFORE policy correctness data are inspected (not generated from observed scores).
THRESHOLD_GRID = sorted(set(
    [round(k / 100, 4) for k in range(20, 100)] + [0.995, 0.9975, 0.999, 0.9995, 1.0]
))
CFG = {
    'evaluation_version': 2,
    'target_error': 0.05,
    'minimum_accepted': 100,
    'minimum_accepted_cameras': 5,
    'policy_familywise_alpha': 0.05,
    'policy_threshold_grid': THRESHOLD_GRID,
    'policy_num_policies': 12,  # 2 architectures * 3 seeds * {raw, scaled}
    'policy_camera_bootstrap_repeats': 10000,
    'policy_camera_bootstrap_seed': 20261104,
    'policy_method': 'fixed-grid binomial Clopper-Pearson Bonferroni selection; approximate simultaneous camera bootstrap is diagnostic only',
    'policy_selection_bound': 'binomial_bonferroni_ucb_95',
    'camera_uncertainty_role': 'diagnostic_only',
    'temperature_bounds': [0.01, 100.0],
    'ece_bins': 15,
    'ece_sensitivity_bins': [10, 20],
    'bootstrap_repeats': 2000,
    'bootstrap_seed': 20261026,
    'test_batch_size': 4,
    'num_workers': 0,
    'checkpoint_rule': 'notebook03 corrected v2 best_selected across head and fine-tuning using event-weighted validation NLL',
    'primary_calibration_gain_metric': 'nll',
}
if not THRESHOLD_GRID or THRESHOLD_GRID[0] != .20 or THRESHOLD_GRID[-1] != 1.0:
    raise ValueError('The predefined threshold grid must span 0.20 to 1.0')

def now():
    return datetime.now(timezone.utc).isoformat()

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def jtext(x):
    return json.dumps(x, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + '\n'

def write(path, text):
    path = Path(path)
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(text, encoding='utf-8')
    temporary.replace(path)

def freeze(path, text):
    if Path(path).exists() and Path(path).read_text(encoding='utf-8') != text:
        raise ValueError(f'Frozen protocol differs: {path}. Do not overwrite an evaluated protocol.')
    if not Path(path).exists():
        write(path, text)

def table(name, rows):
    df = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    write(REPORT / name, df.to_csv(index=False))
    return df

print('Project:', ROOT, '| v2 experiment:', EXPERIMENT_ID)
print('Policy:', len(THRESHOLD_GRID), 'fixed thresholds; binomial nominal familywise 95% under independent-event assumptions.')
print('Camera-bootstrap analysis is approximate screening, not guaranteed camera-aware error control.')
print('CPU | torch:', torch.__version__, '| scipy:', scipy.__version__)


## 2. Verify the v2 data, camera split, selected head-or-fine-tuning checkpoints and development logits

Do not silently use v1 outputs. Verify hashes linking Notebook 01's original camera assignments, Notebook 02's event/image sampling and preprocessing, and Notebook 03's six completed training runs. Check the identity and order of every calibration/policy row and that the two splits share **no camera locations**. No test predictions are computed in this stage. Test manifests are checked for integrity, not used for selecting policies.

Read `training_v2_corrected/training_index.json`; verify completed.json, all three exports,
head/fine-tuning candidate hashes, and the `best_selected.pt` stage/epoch identity.


In [ ]:
required = [EXP / 'protocol.json', EXP / 'selection_protocol.json', EXP / 'classes.json',
            EXP / 'subset_manifest.csv', EXP / 'training_ready_manifest.csv',
            PREPARED / 'camera_split.json', PREPARED / 'preparation_run.json',
            PREPARED / 'image_manifest.csv', PREP_REPORT / 'readiness.json',
            TRAIN_REPORT / 'training_index.json', TRAIN_REPORT / 'training_protocol.json']
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing v2 source file: {path}. Complete Notebooks 01–03 v2 first.')

protocol = json.loads((EXP / 'protocol.json').read_text(encoding='utf-8'))
selection = json.loads((EXP / 'selection_protocol.json').read_text(encoding='utf-8'))
camera_record = json.loads((PREPARED / 'camera_split.json').read_text(encoding='utf-8'))
preparation_record = json.loads((PREPARED / 'preparation_run.json').read_text(encoding='utf-8'))
readiness = json.loads((PREP_REPORT / 'readiness.json').read_text(encoding='utf-8'))
index = json.loads((TRAIN_REPORT / 'training_index.json').read_text(encoding='utf-8'))
train_cfg = json.loads((TRAIN_REPORT / 'training_protocol.json').read_text(encoding='utf-8'))
classes_record = json.loads((EXP / 'classes.json').read_text(encoding='utf-8'))
CLASSES = classes_record['classes']
K = len(CLASSES)
if (protocol.get('pipeline_kind') != 'megadetector_assisted_single_v2'
        or protocol.get('protocol_version') != 2 or protocol.get('experiment_id') != EXPERIMENT_ID):
    raise ValueError('Notebook 04 v2 requires the frozen Notebook 02 v2 image protocol.')
if (selection.get('selection_version') != 2 or selection.get('experiment_id') != EXPERIMENT_ID
        or camera_record.get('protocol') != 'camera_stratified_event_v2'
        or camera_record.get('experiment_id') != EXPERIMENT_ID
        or preparation_record.get('stage') != '01_event_first_camera_split_v2'
        or preparation_record.get('experiment_id') != EXPERIMENT_ID):
    raise ValueError('v2 camera/event sampling provenance is inconsistent.')
if CLASSES != ['coyote', 'deer', 'fox', 'rabbit', 'skunk']:
    raise ValueError('Unexpected experiment classes')
if (protocol['classes'] != selection['classes'] or protocol['classes'] != CLASSES
        or classes_record['class_to_idx'] != {s:i for i,s in enumerate(CLASSES)}
        or selection['class_to_idx'] != classes_record['class_to_idx']):
    raise ValueError('Class order or indices disagree across v2 sources')
if not readiness.get('ready_for_final_training'):
    raise ValueError('Notebook 02 v2 has not reported full image readiness.')
if (readiness.get('protocol_sha256') != sha(EXP/'protocol.json')
        or readiness.get('subset_sha256') != sha(EXP/'subset_manifest.csv')
        or preparation_record.get('split_sha256') != sha(PREPARED/'camera_split.json')
        or preparation_record.get('manifest_sha256') != sha(PREPARED/'image_manifest.csv')):
    raise ValueError('Prepared data do not match frozen v2 upstream records')
for name, expected_sha in protocol['input_sha256'].items():
    relative = Path(name)
    if relative.is_absolute() or '..' in relative.parts:
        raise ValueError(f'Unsafe input provenance path: {name}')
    path = PREPARED / relative
    if not path.is_file() or sha(path) != expected_sha:
        raise ValueError(f'Upstream v2 input hash mismatch: {path}')

if (index.get('experiment_id') != EXPERIMENT_ID or index.get('training_id') != TRAIN_ID
        or not index.get('all_planned_runs_complete')
        or index.get('test_predictions_computed') is not False
        or index['training_sha256'] != sha(TRAIN_REPORT / 'training_protocol.json')
        or index['protocol_sha256'] != sha(EXP / 'protocol.json')
        or index['subset_sha256'] != sha(EXP / 'subset_manifest.csv')
        or index['camera_split_sha256'] != sha(PREPARED / 'camera_split.json')
        or index['selection_protocol_sha256'] != sha(EXP / 'selection_protocol.json')):
    raise ValueError('Notebook 03 v2 training index is incomplete or mismatched')
if (train_cfg.get('experiment_id') != EXPERIMENT_ID or train_cfg.get('training_id') != TRAIN_ID or
        train_cfg.get('ready_manifest_sha256') != sha(EXP / 'training_ready_manifest.csv')):
    raise ValueError('Notebook 03 v2 training configuration does not match the prepared inputs')
if (train_cfg.get('selection') != 'lowest_validation_NLL_across_head_and_finetune'
        or train_cfg.get('selection_metric') != 'validation_nll'
        or train_cfg.get('selection_weighting') != 'event'
        or train_cfg.get('final_checkpoint_file') != 'best_selected.pt'):
    raise ValueError('Notebook 04 requires corrected Notebook 03 event-NLL cross-stage selection.')
for package, version in [('torch', str(torch.__version__)), ('torchvision', str(torchvision.__version__))]:
    if train_cfg['versions'][package] != version:
        raise ValueError(f'Use the Notebook 03 v2 environment: {package} was '
                         f'{train_cfg["versions"][package]}, currently {version}')

manifest = pd.read_csv(EXP / 'training_ready_manifest.csv', dtype=str, keep_default_na=False)
base = pd.read_csv(EXP / 'subset_manifest.csv', dtype=str, keep_default_na=False)
source_events = pd.read_csv(PREPARED / 'image_manifest.csv', dtype=str, keep_default_na=False)
if not set(base.columns).issubset(manifest.columns) or not manifest[base.columns].equals(base):
    raise ValueError('Ready manifest differs from the frozen v2 subset')
if (not manifest.image_id.is_unique or not manifest.seq_id.is_unique or
        not source_events.image_id.is_unique or not source_events.seq_id.is_unique):
    raise ValueError('Event or image IDs are repeated')
if set(manifest.split) != {'train','validation','calibration','policy','test'}:
    raise ValueError('A required v2 split is empty')
if not manifest.groupby('location').split.nunique().eq(1).all():
    raise ValueError('One camera appears in more than one split')
if not manifest.location.map(camera_record['assignments']).eq(manifest.split).all():
    raise ValueError('Selected v2 camera labels conflict with Notebook 01')
if not manifest.image_id.isin(source_events.image_id).all():
    raise ValueError('Selected image is absent from Notebook 01')
manifest['label'] = manifest.label.astype(int)
if not manifest.label.eq(manifest.class_name.map({c:i for i,c in enumerate(CLASSES)})).all():
    raise ValueError('Label and category mismatch')
for col in ['source_sha256','cache_sha256']:
    if manifest[col].eq('').any() or not manifest.groupby(col).split.nunique().le(1).all():
        raise ValueError(f'Missing or reused cross-split image hash: {col}')

expected = {(m,int(s)) for m in train_cfg['models'] for s in train_cfg['planned_seeds']}
RUNS = index['completed_runs']
if (len(expected) != 6 or len(RUNS) != 6 or
        {(r['model'],int(r['seed'])) for r in RUNS} != expected):
    raise ValueError('All six independent v2 model/seed runs are required for this final evaluation')
if CFG['policy_num_policies'] != 2 * len(RUNS):
    raise ValueError('Multiple-testing correction does not cover every raw/scaled candidate policy')


def safe_path(relative):
    p = (ROOT / relative).resolve()
    if not p.is_relative_to(ROOT):
        raise ValueError('Path leaves v2 project root')
    return p


def check_arrays(a, frame):
    if a['logits'].shape != (len(frame), K) or not np.isfinite(a['logits']).all():
        raise ValueError('Invalid logits shape or nonfinite values')
    for key, col, dtype in [('image_ids','image_id',str),('labels','label',np.int64),
                            ('locations','location',str),('seq_ids','seq_id',str)]:
        if not np.array_equal(a[key], frame[col].to_numpy(dtype=dtype)):
            raise ValueError(f'Export identity differs from v2 manifest: {key}')
    if not np.array_equal(a['class_names'],np.asarray(CLASSES)):
        raise ValueError('Export class order differs')


def load_development(run, split):
    run_dir = safe_path(run['run_relative_path'])
    entry = run['exports'][split]
    p = run_dir / entry['file']
    if p.parent != run_dir or not p.is_file() or sha(p) != entry['sha256']:
        raise ValueError(f'Invalid saved development export: {p}')
    with np.load(p, allow_pickle=False) as saved:
        a = {k:saved[k].copy() for k in saved.files}
    frame = manifest.loc[manifest.split.eq(split)].reset_index(drop=True)
    check_arrays(a, frame)
    for key, value in [('checkpoint_sha256',run['checkpoint_sha256']),
                       ('training_sha256',index['training_sha256']),
                       ('subset_sha256',index['subset_sha256']),
                       ('protocol_sha256',index['protocol_sha256']),
                       ('camera_split_sha256',index['camera_split_sha256']),
                       ('experiment_id',EXPERIMENT_ID),
                       ('model_name',run['model']),('seed',int(run['seed']))]:
        if a[key].item() != value:
            raise ValueError(f'Development logits provenance mismatch: {key}')
    return a

def checkpoint_provenance(run):
    return {'training_sha256':index['training_sha256'],
            'subset_sha256':index['subset_sha256'], 'protocol_sha256':index['protocol_sha256'],
            'camera_split_sha256':index['camera_split_sha256'],
            'selection_protocol_sha256':index['selection_protocol_sha256'],
            'ready_manifest_sha256':train_cfg['ready_manifest_sha256'],
            'experiment_id':EXPERIMENT_ID, 'training_id':TRAIN_ID,
            'classes':CLASSES, 'model_name':run['model'], 'seed':int(run['seed']), 'pilot_only':False}


def verify_run_record(run):
    run_dir = safe_path(run['run_relative_path'])
    if (run.get('checkpoint_file') != 'best_selected.pt'
            or run.get('selected_stage') not in ('head', 'finetune')
            or not isinstance(run.get('selected_epoch'), int) or run['selected_epoch'] < 1):
        raise ValueError('Missing corrected Notebook 03 selected checkpoint identity')
    if set(run.get('exports', {})) != {'validation', 'calibration', 'policy'}:
        raise ValueError('All three Notebook 03 development exports are required')
    completed = json.loads((run_dir / 'completed.json').read_text(encoding='utf-8'))
    for field, value in checkpoint_provenance(run).items():
        if completed.get(field) != value:
            raise ValueError(f'Completed-run provenance differs ({field})')
    for field in ('checkpoint_file', 'checkpoint_sha256', 'selected_stage', 'selected_epoch', 'exports'):
        if completed.get(field) != run[field]:
            raise ValueError(f'Training index differs from completed.json ({field})')
    if completed.get('status') != 'completed':
        raise ValueError('Notebook 03 run is not completed')
    candidates = completed.get('candidate_checkpoints', {})
    if set(candidates) != {'head', 'finetune'}:
        raise ValueError('Missing cross-stage candidate provenance')
    for phase, entry in candidates.items():
        candidate = run_dir / f'best_{phase}.pt'
        if (entry.get('file') != candidate.name or not candidate.is_file()
                or sha(candidate) != entry.get('sha256')):
            raise ValueError(f'Candidate checkpoint missing/changed: {phase}')
    path = run_dir / run['checkpoint_file']
    if not path.is_file() or sha(path) != run['checkpoint_sha256']:
        raise ValueError('Selected checkpoint hash differs from the corrected training index')
    return completed


DEV = {}
RUN_RECORDS = {}
for run in RUNS:
    key = (run['model'], int(run['seed']))
    completed = verify_run_record(run)
    RUN_RECORDS[key] = completed
    checked = {split:load_development(run, split) for split in ['validation', 'calibration', 'policy']}
    DEV[key] = {split:checked[split] for split in ['calibration', 'policy']}
    print('Verified:', *key, '| selected:', run['selected_stage'], run['selected_epoch'])

# Freeze the statistical method and all input identities *before* test inference.
EVAL_INPUT = {
    'settings':CFG, 'experiment_id':EXPERIMENT_ID, 'training_id':TRAIN_ID,
    'training_sha256':index['training_sha256'],
    'protocol_sha256':index['protocol_sha256'],
    'selection_protocol_sha256':index['selection_protocol_sha256'],
    'camera_split_sha256':index['camera_split_sha256'],
    'subset_sha256':index['subset_sha256'],
    'ready_manifest_sha256':sha(EXP/'training_ready_manifest.csv'),
    'runs':[{'model':r['model'],'seed':int(r['seed']),'checkpoint_file':r['checkpoint_file'],
             'selected_stage':r['selected_stage'],'selected_epoch':r['selected_epoch'],
             'checkpoint_sha256':r['checkpoint_sha256'],
             'development_exports':r['exports']} for r in RUNS],
    'versions':{'python':sys.version.split()[0],'torch':str(torch.__version__),
                'torchvision':str(torchvision.__version__), 'numpy':np.__version__,
                'pandas':pd.__version__, 'pillow':PIL.__version__, 'scipy':scipy.__version__},
}
freeze(REPORT/'evaluation_protocol.json',jtext(EVAL_INPUT))
EVAL_SHA = sha(REPORT/'evaluation_protocol.json')
print('Frozen v2 evaluation protocol:',EVAL_SHA)
print('Camera counts:',manifest.groupby('split').location.nunique().to_dict())


## 3. Metric definitions and temperature fitting
NLL uses a stable log-softmax calculation. Accuracy and macro-F1 use the original logits' argmax, avoiding numerical tie changes after extreme scaling. ECE is the image-weighted absolute difference between mean confidence and accuracy in each confidence bin; empty bins contribute zero. ECE depends on binning and is not a complete calibration measure.

We optimize inverse temperature $\beta=1/T$. NLL is convex in this scalar. A deterministic derivative bisection finds the bounded optimum in $T\in[0.01,100]$, including endpoint checks; $T=1$ is retained if numerical rounding would worsen the calibration objective. Boundary solutions are flagged, not silently hidden. These bounds are fixed before testing. Calibration-split improvement is fitted performance, not independent evidence of generalization.

For $N$ evaluated events and $K=5$ categories, let $p_{ik}(T)$ be the temperature-scaled
softmax probability and $y_i$ the reference category. The probability losses are

$$\operatorname{NLL}(T)=-\frac{1}{N}\sum_{i=1}^{N}\log p_{i,y_i}(T),\qquad
\operatorname{Brier}(T)=\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{K}
\left(p_{ik}(T)-\mathbf{1}\{y_i=k\}\right)^2.$$

Natural logarithms are used. Brier is **summed across categories**, then averaged across events;
it is not additionally divided by five. Lower NLL, Brier and ECE are better. Macro-F1 weights
each of the five categories equally; a zero F1 denominator contributes zero. ECE uses 15 fixed
equal-width confidence bins, with 10- and 20-bin sensitivity checks. Improved NLL need not imply
improved ECE or increased selective coverage. Accuracy/F1 use all evaluated events; accepted
error uses only accepted events.


In [ ]:
def probabilities(logits,T=1.0):
    z=np.asarray(logits,dtype=np.float64)/float(T)
    z-=z.max(axis=1,keepdims=True)
    logp=z-np.log(np.exp(z).sum(axis=1,keepdims=True))
    return np.exp(logp),logp

def reliability(logits,y,T,bins):
    p,_=probabilities(logits,T);conf=p.max(1);correct=logits.argmax(1)==y
    b=np.minimum((conf*bins).astype(int),bins-1)
    count=np.bincount(b,minlength=bins)
    cs=np.bincount(b,weights=conf,minlength=bins)
    hits=np.bincount(b,weights=correct,minlength=bins)
    return pd.DataFrame({"bin":np.arange(bins),"lower":np.arange(bins)/bins,
        "upper":(np.arange(bins)+1)/bins,"count":count,
        "confidence":np.divide(cs,count,out=np.full(bins,np.nan),where=count>0),
        "accuracy":np.divide(hits,count,out=np.full(bins,np.nan),where=count>0)})

def metrics(logits,y,T=1.0):
    p,lp=probabilities(logits,T);pred=logits.argmax(1)
    cm=np.bincount(y*K+pred,minlength=K*K).reshape(K,K)
    den=cm.sum(0)+cm.sum(1)
    f1=np.divide(2*np.diag(cm),den,out=np.zeros(K,dtype=float),where=den>0)
    rel=reliability(logits,y,T,CFG["ece_bins"])
    return {"nll":float(-lp[np.arange(len(y)),y].mean()),
        "accuracy":float((pred==y).mean()),"macro_f1":float(f1.mean()),
        "brier":float(((p-np.eye(K)[y])**2).sum(1).mean()),
        "ece":float((rel["count"]*(rel.confidence-rel.accuracy).abs()).sum()/len(y))}

def fit_temperature(logits,y):
    z=np.asarray(logits,dtype=np.float64)
    loT,hiT=CFG["temperature_bounds"];lo,hi=1/hiT,1/loT
    def derivative(beta):
        p,_=probabilities(z,1/beta)
        return float(((p*z).sum(1)-z[np.arange(len(y)),y]).mean())
    if derivative(lo)>=0: beta=lo
    elif derivative(hi)<=0: beta=hi
    else:
        for _ in range(90):
            mid=(lo+hi)/2
            if derivative(mid)>0:hi=mid
            else:lo=mid
        beta=(lo+hi)/2
    T=1/beta
    raw=metrics(z,y,1)["nll"];scaled=metrics(z,y,T)["nll"]
    if scaled>raw: T=1.0;scaled=raw
    return {"temperature":float(T),"calibration_nll_raw":raw,"calibration_nll_scaled":scaled,
            "at_temperature_boundary":bool(np.isclose(T,loT) or np.isclose(T,hiT))}

TEMPERATURES={};temperature_rows=[]
for key,data in DEV.items():
    a=data["calibration"];result=fit_temperature(a["logits"],a["labels"])
    TEMPERATURES[key]=result["temperature"]
    temperature_rows.append({"model":key[0],"seed":key[1],**result})
temperature_summary=table("temperature_summary.csv",temperature_rows)
display(temperature_summary)


## 4. Fixed-grid binomial screening and separate camera diagnostics
**What is being bounded?** For a fixed model, seed, raw/scaled variant and confidence threshold
$t$, accept a policy event when its maximum softmax probability is at least $t$. Let $n(t)$ be
the number accepted, $e(t)$ the number of incorrect accepted predictions, and $k(t)$ the number
of cameras with at least one accepted event. Observed accepted error is $e(t)/n(t)$ when $n(t)>0$.
This is not the error rate over all policy images or a per-category bound.

**Binomial bound and multiplicity.** The nominal familywise error budget is $\delta=0.05$.
There are $M=2\times3\times2\times85=1020$ predefined comparisons: two architectures,
three seeds, raw/scaled probabilities and 85 thresholds. Each uses

$$\alpha_* = \frac{\delta}{M}=\frac{0.05}{1020}\approx4.90196\times10^{-5}.$$

For $n>0$, the one-sided Clopper–Pearson upper bound is

$$U_{\mathrm{bin}}(e,n)=
\begin{cases}
F^{-1}_{\mathrm{Beta}(e+1,n-e)}(1-\alpha_*), & 0\le e<n,\\
1, & e=n.
\end{cases}$$

Here $F^{-1}$ is the beta-distribution quantile. For $n=0$, accepted error and its bound are
undefined. This is a **one-sided** bound, so $\alpha_*$ is not divided by two. Bonferroni allows
searching the fixed grid while retaining nominal 95% simultaneous coverage **under the binomial
sampling assumptions**. The threshold/model tests need not be independent of one another;
independence of accepted events within each binomial model is the separate assumption at issue.
Within-camera dependence can violate it. Neither the bound's name nor its `_95` output suffix
establishes 95% camera-aware coverage, per-camera control or a guarantee under deployment shift.

**Zero errors can pass.** For $e=0$, the same exact formula simplifies to

$$U_{\mathrm{bin}}(0,n)=1-\alpha_*^{1/n}.$$

These are mathematical examples under the fixed settings, **not experimental results**:

| Accepted policy events, all correct | Bonferroni-adjusted binomial upper bound |
| --- | ---: |
| 100 | 9.45% |
| 194 | 4.99% |
| 200 | 4.84% |
| 1,500 | 0.66% |

At least 194 error-free accepted events are therefore needed for this bound to be at most 5%;
the unchanged camera-support requirement must also hold. Zero observed errors means observed
accepted error is zero, not that the unknown error rate or uncertainty is zero. Zero accepted
events is a different case: its accepted error is undefined.

**Selection rule.** A candidate passes if and only if its finite $U_{\mathrm{bin}}\le0.05$,
$n\ge100$ and $k\ge5$. These image/camera floors are support checks, not proof of independence
or sufficient precision for new-camera inference. Select the greatest accepted count, breaking
ties with the higher cutoff. If none passes, report `no_qualifying_threshold` and no automatic
acceptance threshold. Raw and scaled variants are screened separately; no test outcomes choose
the threshold.

**Camera-bootstrap diagnostic.** Keep all events from a camera together. The existing 10,000
seeded whole-policy-camera resamples are shared across every cutoff, model, seed and variant.
Each supported candidate's resampled accepted error is computed from its resampled error and
acceptance counts. A resample accepting nothing retains the original 100%-risk assignment for
this diagnostic; that assignment is not an observed error rate. Let $s_j$ be the sample SD of
resampled risks. Across candidates with available estimates, the shared critical value $q$ is
the 95th percentile of the maximum positive standardized deviation
$(r_j^*-\widehat r_j)/s_j$. The reported approximate upper estimate is
$\min(1,\widehat r_j+q s_j)$. This simultaneous standardized-bootstrap calculation is
approximate and **does not select thresholds**.

The diagnostic is unavailable for zero observed errors, nonfinite resampled risks or SD, SD at
most $10^{-12}$, or a missing shared critical value. Resampling zero errors cannot create unseen
mistakes; even artificial variation from empty-acceptance resamples does not supply that missing
information. A recorded SD of zero describes the resamples, not certainty about future cameras.
Unavailable upper estimates remain missing (`NaN` in tables or JSON `null`), never a substituted
zero. Finite estimates with few informative cameras also do not establish valid 95% camera-aware
coverage. Three training seeds reuse the same policy cameras and do not triple the camera sample.

**Reading the outputs.** Candidate CSVs retain `camera_bootstrap_sd`,
`camera_uncertainty_available`, `camera_bootstrap_degenerate`, the reason, and
`camera_bootstrap_simultaneous_upper_approx`. Selected policies retain camera diagnostics even
when the estimate is unavailable or exceeds 5%. That combination with `status=selected` is
intentional. For compatibility, `screening_upper` aliases `binomial_bonferroni_ucb_95`;
`passes_screening_5pct` applies only that binomial bound and the support checks. No maximum of
binomial and camera bounds is used. Temperatures, decisions, diagnostic reasons and the amended
method are frozen before final test inference.

References: [NIST: exact binomial confidence bounds](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm),
[Learn then Test: risk control and multiple testing assumptions](https://arxiv.org/html/2110.01052v5),
and [Cameron and Miller: cluster inference and few-cluster limitations](https://faculty.econ.ucdavis.edu/faculty/cameron/research/Cameron_Miller_JHR_2014_July_09.pdf).


In [ ]:
# Binomial familywise confidence applies only under independent-event assumptions.
# The camera analysis is approximate and diagnostic only; it does not select thresholds.
N_THRESHOLDS = len(CFG['policy_threshold_grid'])
POLICY_FAMILY_SIZE = CFG['policy_num_policies'] * N_THRESHOLDS
ALPHA_PER_CANDIDATE = CFG['policy_familywise_alpha'] / POLICY_FAMILY_SIZE
if not (0 < ALPHA_PER_CANDIDATE < 1):
    raise ValueError('Invalid policy multiple-testing correction')


def clopper_pearson_upper(errors, n, alpha=ALPHA_PER_CANDIDATE):
    """Exact one-sided binomial upper bound at alpha for a fixed, independent-event policy."""
    errors, n = int(errors), int(n)
    if not (0 <= errors <= n):
        raise ValueError('Invalid error/acceptance counts')
    if n == 0:
        return np.nan
    if errors == n:
        return 1.0
    return float(beta_distribution.ppf(1.0-alpha, errors+1, n-errors))


def camera_bootstrap_analysis(candidates, counts_a, counts_e, weights):
    """Never treat degenerate camera resampling as evidence of zero uncertainty."""
    supported = candidates.minimum_support_met.to_numpy(dtype=bool)
    size = len(candidates)
    sd = np.full(size, np.nan)
    reasons = np.full(size, 'insufficient_policy_support', dtype=object)
    available = np.zeros(size, dtype=bool)
    roots = np.zeros((len(weights), size), dtype=float)
    if len(weights) < 2:
        raise ValueError('Camera bootstrap needs at least two replicates')
    if supported.any():
        total_a = weights @ counts_a[supported].T
        total_e = weights @ counts_e[supported].T
        # Preserve the existing conservative treatment of empty bootstrap acceptance.
        risk = np.divide(total_e, total_a, out=np.ones_like(total_a, dtype=float), where=total_a>0)
        observed = candidates.loc[supported, 'accepted_error'].to_numpy(dtype=float)
        spread = np.std(risk, axis=0, ddof=1)
        sd[supported] = spread
        positions = np.flatnonzero(supported)
        for j, i in enumerate(positions):
            # Empty-acceptance replicates assigned risk=1 cannot identify unseen error risk.
            if int(candidates.iloc[i]['errors']) == 0:
                reasons[i] = 'no_observed_errors_camera_uncertainty_unidentified'
            elif not np.isfinite(spread[j]) or not np.isfinite(risk[:, j]).all():
                reasons[i] = 'nonfinite_camera_bootstrap_estimate'
            elif spread[j] <= 1e-12:
                reasons[i] = 'degenerate_camera_bootstrap_variation'
            else:
                available[i] = True
                reasons[i] = 'approximate_estimate_available'
                roots[:, i] = np.maximum(0.0, (risk[:, j]-observed[j])/spread[j])
    return {'sd':sd, 'available':available, 'reason':reasons, 'roots':roots}


def simultaneous_camera_critical(analyses, alpha):
    maxima = None
    for analysis in analyses.values():
        if analysis['available'].any():
            local = analysis['roots'][:, analysis['available']].max(axis=1)
            maxima = local if maxima is None else np.maximum(maxima, local)
    if maxima is None:
        return None  # Insufficient evidence; never substitute critical=0.
    critical = float(np.quantile(maxima, 1-alpha, method='higher'))
    return critical if np.isfinite(critical) else None


def add_camera_screening_bounds(candidates, analysis, critical):
    out = candidates.copy()
    out['camera_bootstrap_sd'] = analysis['sd']
    available = analysis['available'].copy()
    reasons = analysis['reason'].copy()
    if critical is None or not np.isfinite(critical):
        reasons[available] = 'simultaneous_camera_estimate_unavailable'
        available[:] = False
    upper = np.full(len(out), np.nan)
    if available.any():
        upper[available] = np.minimum(1.0, out.loc[available, 'accepted_error'].to_numpy(dtype=float)
                                     + critical*analysis['sd'][available])
    out['camera_uncertainty_available'] = available
    out['camera_uncertainty_reason'] = reasons
    out['camera_bootstrap_degenerate'] = out.minimum_support_met & ~available
    out['camera_bootstrap_simultaneous_upper_approx'] = upper
    # Preserve the output names used downstream; selection uses only the binomial bound.
    # Missing/high camera evidence remains visible but never vetoes a candidate.
    out['screening_upper'] = out['binomial_bonferroni_ucb_95']
    out['passes_screening_5pct'] = (out.minimum_support_met
        & np.isfinite(out.screening_upper) & out.screening_upper.le(CFG['target_error']))
    return out


def choose_screening_policy(candidates):
    passing = candidates.loc[candidates.passes_screening_5pct]
    if passing.empty:
        return {'status':'no_qualifying_threshold', 'reason':'no_threshold_passes_binomial_screening',
                'threshold':None, 'accepted':0, 'errors':0, 'accepted_cameras':0,
                'coverage':0.0, 'accepted_error':None, 'binomial_bonferroni_ucb_95':None,
                'camera_uncertainty_available':False, 'camera_uncertainty_reason':'no_threshold_selected',
                'camera_bootstrap_degenerate':None, 'camera_bootstrap_sd':None,
                'camera_bootstrap_simultaneous_upper_approx':None, 'screening_upper':None}
    chosen = passing.sort_values(['accepted', 'threshold'], ascending=[False, False]).iloc[0]
    camera_upper = chosen.camera_bootstrap_simultaneous_upper_approx
    camera_sd = chosen.camera_bootstrap_sd
    return {'status':'selected', 'reason':'passes_binomial_bonferroni_screening',
            'threshold':float(chosen.threshold), 'accepted':int(chosen.accepted),
            'errors':int(chosen.errors), 'accepted_cameras':int(chosen.accepted_cameras),
            'coverage':float(chosen.coverage), 'accepted_error':float(chosen.accepted_error),
            'binomial_bonferroni_ucb_95':float(chosen.binomial_bonferroni_ucb_95),
            'camera_uncertainty_available':bool(chosen.camera_uncertainty_available),
            'camera_uncertainty_reason':str(chosen.camera_uncertainty_reason),
            'camera_bootstrap_degenerate':bool(chosen.camera_bootstrap_degenerate),
            'camera_bootstrap_sd':float(camera_sd) if np.isfinite(camera_sd) else None,
            'camera_bootstrap_simultaneous_upper_approx':float(camera_upper) if np.isfinite(camera_upper) else None,
            'screening_upper':float(chosen.screening_upper)}


policy_frame = manifest.loc[manifest.split.eq('policy')].reset_index(drop=True)
policy_cameras = np.unique(policy_frame.location.to_numpy(dtype=str))
N_POLICY_CAMERAS = len(policy_cameras)
if N_POLICY_CAMERAS < CFG['minimum_accepted_cameras']:
    raise ValueError('Too few represented policy cameras for the stipulated rule')
policy_cam_idx = np.searchsorted(policy_cameras, policy_frame.location.to_numpy(dtype=str))
rng_policy = np.random.default_rng(CFG['policy_camera_bootstrap_seed'])
POLICY_CAMERA_W = rng_policy.multinomial(
    N_POLICY_CAMERAS, np.full(N_POLICY_CAMERAS, 1.0/N_POLICY_CAMERAS),
    size=CFG['policy_camera_bootstrap_repeats']).astype(np.float64)


def policy_candidates(logits, y, locations, T):
    p, _ = probabilities(logits, T)
    confidences = p.max(axis=1)
    mistakes = (np.asarray(logits).argmax(1) != np.asarray(y))
    locations = np.asarray(locations, dtype=str)
    if len(confidences) != len(policy_frame) or not np.array_equal(locations, policy_frame.location):
        raise ValueError('Policy export does not match the v2 policy image/camera order')
    accept = confidences[None, :] >= np.asarray(CFG['policy_threshold_grid'])[:, None]
    errors = accept & mistakes[None, :]
    per_camera_accepted = np.zeros((N_THRESHOLDS, N_POLICY_CAMERAS), dtype=np.int64)
    per_camera_errors = np.zeros_like(per_camera_accepted)
    for ci in range(N_POLICY_CAMERAS):
        mask = policy_cam_idx == ci
        per_camera_accepted[:, ci] = accept[:, mask].sum(axis=1)
        per_camera_errors[:, ci] = errors[:, mask].sum(axis=1)
    n = per_camera_accepted.sum(axis=1)
    e = per_camera_errors.sum(axis=1)
    accepted_cameras = (per_camera_accepted>0).sum(axis=1)
    point = np.divide(e, n, out=np.full(N_THRESHOLDS, np.nan), where=n>0)
    eligible = ((n >= CFG['minimum_accepted']) & (accepted_cameras >= CFG['minimum_accepted_cameras']))
    candidates = pd.DataFrame({'threshold':CFG['policy_threshold_grid'], 'accepted':n, 'errors':e,
        'accepted_cameras':accepted_cameras, 'coverage':n/len(y), 'accepted_error':point,
        'empirical_error_le_target':(n>0)&(point<=CFG['target_error']), 'minimum_support_met':eligible,
        'binomial_bonferroni_ucb_95':[clopper_pearson_upper(x, m) for x, m in zip(e, n)]})
    return candidates, per_camera_accepted, per_camera_errors


POLICY_CANDIDATES = {}
POLICY_COUNTS = {}
POLICY_CAMERA_ANALYSES = {}
for run in RUNS:
    key = (run['model'], int(run['seed']))
    a = DEV[key]['policy']
    for variant, T in [('raw', 1.0), ('scaled', TEMPERATURES[key])]:
        identifier = (*key, variant)
        candidates, counts_a, counts_e = policy_candidates(a['logits'], a['labels'], a['locations'], T)
        POLICY_CANDIDATES[identifier] = candidates
        POLICY_COUNTS[identifier] = (counts_a, counts_e)
        POLICY_CAMERA_ANALYSES[identifier] = camera_bootstrap_analysis(
            candidates, counts_a, counts_e, POLICY_CAMERA_W)
if len(POLICY_CANDIDATES) != CFG['policy_num_policies']:
    raise ValueError('Not all twelve v2 policy variants were evaluated')

POLICY_MAX_T_CRITICAL = simultaneous_camera_critical(POLICY_CAMERA_ANALYSES, CFG['policy_familywise_alpha'])
POLICY_CRITICAL_TEXT = ('unavailable (insufficient camera evidence)' if POLICY_MAX_T_CRITICAL is None
                        else f'{POLICY_MAX_T_CRITICAL:.5f}')
POLICIES = {}
policy_rows = []
frozen_runs = []
for run in RUNS:
    key = (run['model'], int(run['seed']))
    policies = {}
    for variant, T in [('raw', 1.0), ('scaled', TEMPERATURES[key])]:
        identifier = (*key, variant)
        candidates = add_camera_screening_bounds(POLICY_CANDIDATES[identifier],
            POLICY_CAMERA_ANALYSES[identifier], POLICY_MAX_T_CRITICAL)
        result = choose_screening_policy(candidates)
        policies[variant] = result
        table(f'policy_candidates_{key[0]}_seed{key[1]}_{variant}.csv', candidates)
        policy_rows.append({'model':key[0], 'seed':key[1], 'variant':variant, 'temperature':T, **result})
        print(f'{key[0]} seed {key[1]} {variant}: status={result["status"]}; '
              f'{int(candidates.camera_bootstrap_degenerate.sum())} supported cutoffs lack a camera diagnostic (not a selection gate)')
    POLICIES[key] = policies
    frozen_runs.append({'model':key[0], 'seed':key[1], 'checkpoint_sha256':run['checkpoint_sha256'],
                        'temperature':TEMPERATURES[key], 'policies':policies})

freeze(REPORT/'frozen_calibration_and_policies.json', jtext({
    'evaluation_sha256':EVAL_SHA, 'policy_method':CFG['policy_method'],
    'policy_selection_bound':CFG['policy_selection_bound'],
    'camera_uncertainty_role':CFG['camera_uncertainty_role'],
    'familywise_alpha':CFG['policy_familywise_alpha'], 'cp_single_test_alpha':ALPHA_PER_CANDIDATE,
    'policy_camera_count':N_POLICY_CAMERAS, 'policy_max_t_critical':POLICY_MAX_T_CRITICAL,
    'camera_interpretation':'approximate_diagnostic_not_guaranteed_camera_coverage',
    'policy_bootstrap_repeats':CFG['policy_camera_bootstrap_repeats'], 'runs':frozen_runs}))
DECISION_SHA = sha(REPORT/'frozen_calibration_and_policies.json')
policy_summary = table('policy_summary.csv', policy_rows)
display(policy_summary)
print('Policy decisions frozen before test inference:', DECISION_SHA)


## 5. Final held-out test forward passes (no parameter fitting)

Use only the corrected v2 `best_selected.pt` checkpoints and frozen v2 test image cache. Recheck test-input SHA-256 hashes, PNG size/format, labels and provenance. Predictions are computed with `model.eval()` and `torch.inference_mode()` using the same preprocessing as Notebook 03 v2. Tests are not used in temperature fitting or threshold selection. Cache test logits with the **evaluation and frozen decision hashes** so previously evaluated results cannot be silently paired with a different policy.

Recheck the checkpoint SHA-256 immediately before loading model weights, then verify loaded provenance.

**Immutable prediction hashes:** `test_exports.json` now journals each newly generated file's
creation SHA-256 immediately. Cache reuse checks that original digest and loads the same verified
byte snapshot. Existing entries are never replaced by newly computed cache hashes. Legacy complete
registries remain usable when hashes and provenance match. If a cache has no original hash record,
stop: restore the original registry or regenerate in a separate evaluation cache. Do not manually
adopt its current hash. A completed evaluation also anchors the registry hash in `completion.json`.
Interruptions before a file's hash is journalled leave an unverified cache that must not be silently
reused. All critical integrity/provenance checks use explicit exceptions, including optimized Python.


In [ ]:
def verify_test_image(row):
    path = safe_path(row.cache_relative_path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing test input: {row.image_id}')
    if sha(path) != row.cache_sha256:
        raise ValueError(f'Test image hash differs from the frozen manifest: {row.image_id}')
    with Image.open(path) as picture:
        if picture.size != (224, 224) or picture.mode != 'RGB' or picture.format != 'PNG':
            raise ValueError(f'Unexpected test image dimensions/mode/format: {row.image_id}')
        picture.load()


test_frame = manifest.loc[manifest.split.eq('test')].reset_index(drop=True)
if len(test_frame) == 0:
    raise ValueError('The frozen test set is empty')
for i, row in enumerate(test_frame.itertuples(index=False), 1):
    verify_test_image(row)
    if i % 500 == 0:
        print(f'Verified {i}/{len(test_frame)} test inputs', flush=True)

class TestDataset(Dataset):
    transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    def __len__(self):return len(test_frame)
    def __getitem__(self,i):
        row=test_frame.iloc[i]
        with Image.open(safe_path(row.cache_relative_path)) as im: x=self.transform(im.convert("RGB"))
        return x,row.image_id

def build_model(name):
    if name=="mobilenet_v3_large": model=models.mobilenet_v3_large(weights=None)
    elif name=="efficientnet_b0": model=models.efficientnet_b0(weights=None)
    else: raise ValueError(name)
    model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,K)
    return model

def load_verified_checkpoint(run):
    path = safe_path(run['run_relative_path']) / run['checkpoint_file']
    expected = checkpoint_provenance(run)
    if run['checkpoint_file'] != 'best_selected.pt':
        raise ValueError('Inference requires the corrected selected checkpoint')
    if not path.is_file() or sha(path) != run['checkpoint_sha256']:
        raise ValueError('Checkpoint hash changed immediately before loading weights')
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    for field, value in expected.items():
        if checkpoint.get(field) != value:
            raise ValueError(f'Loaded checkpoint identity/provenance differs ({field})')
    if (checkpoint.get('phase') != run['selected_stage'] or checkpoint.get('epoch') != run['selected_epoch']
            or checkpoint.get('source_checkpoint_file') != f"best_{run['selected_stage']}.pt"
            or checkpoint.get('source_checkpoint_sha256') != RUN_RECORDS[(run['model'], int(run['seed']))]
                ['candidate_checkpoints'][run['selected_stage']]['sha256']):
        raise ValueError('Loaded selected stage/epoch/source differs from Notebook 03')
    return checkpoint


def test_prediction_provenance(run):
    return {'checkpoint_sha256':run['checkpoint_sha256'], 'evaluation_sha256':EVAL_SHA,
            'decisions_sha256':DECISION_SHA, 'experiment_id':EXPERIMENT_ID,
            'model_name':run['model'], 'seed':int(run['seed'])}


def prediction_destination(run):
    return CACHE / f"{run['model']}_seed{int(run['seed'])}_test_logits.npz"


def validate_test_export_record(record):
    if (not isinstance(record, dict) or record.get('evaluation_sha256') != EVAL_SHA
            or record.get('decisions_sha256') != DECISION_SHA
            or not isinstance(record.get('exports'), list)):
        raise ValueError('Test-export registry has invalid or mismatching evaluation/decision provenance')
    expected = {(run['model'], int(run['seed'])):run for run in RUNS}
    entries = {}
    for entry in record['exports']:
        if not isinstance(entry, dict) or not isinstance(entry.get('seed'), int):
            raise ValueError('Malformed test-export registry entry')
        key = (entry.get('model'), entry['seed'])
        if key not in expected or key in entries:
            raise ValueError('Duplicate or unexpected model/seed in test-export registry')
        digest = entry.get('sha256')
        if (not isinstance(digest, str) or len(digest) != 64
                or any(c not in '0123456789abcdef' for c in digest)):
            raise ValueError('Missing or invalid original test-prediction hash')
        if not isinstance(entry.get('file'), str) or safe_path(entry['file']) != prediction_destination(expected[key]).resolve():
            raise ValueError('Test-export registry path differs from the expected prediction file')
        entries[key] = entry
    return entries


def load_test_export_registry():
    path = REPORT / 'test_exports.json'
    completion_path = REPORT / 'completion.json'
    sealed = completion_path.exists()
    if sealed:
        completion = json.loads(completion_path.read_text(encoding='utf-8'))
        if (completion.get('evaluation_sha256') != EVAL_SHA
                or completion.get('decisions_sha256') != DECISION_SHA):
            raise ValueError('Existing completion record belongs to a different evaluation')
        expected_hash = completion.get('outputs', {}).get('test_exports.json')
        if not path.is_file() or not expected_hash or sha(path) != expected_hash:
            raise ValueError('Original test-export registry is missing or differs from the completion record')
    if path.exists():
        if not path.is_file():
            raise ValueError('The test-export registry is not a regular file')
        payload = path.read_bytes()
        record = json.loads(payload.decode('utf-8'))
        fingerprint = hashlib.sha256(payload).hexdigest()
        if sealed and fingerprint != expected_hash:
            raise ValueError('Original registry bytes differ from the completion-record hash')
    else:
        record = {'evaluation_sha256':EVAL_SHA, 'decisions_sha256':DECISION_SHA, 'exports':[]}
        fingerprint = None
    entries = validate_test_export_record(record)
    if sealed and set(entries) != {(r['model'], int(r['seed'])) for r in RUNS}:
        raise ValueError('Completed evaluation does not register every planned test export')
    return {'path':path, 'record':record, 'entries':entries, 'fingerprint':fingerprint, 'sealed':sealed}


def verify_test_export_registry_unchanged(state):
    path, fingerprint = state['path'], state['fingerprint']
    if fingerprint is None:
        if path.exists():
            raise ValueError('A test-export registry appeared during inference; stop rather than overwrite it')
    elif not path.is_file() or sha(path) != fingerprint:
        raise ValueError('The test-export registry changed during inference; original records were not overwritten')


def load_cached_test_predictions(run, state, frame):
    verify_test_export_registry_unchanged(state)
    key = (run['model'], int(run['seed']))
    entry = state['entries'].get(key)
    if entry is None:
        raise ValueError('Cached test predictions have no recorded original hash. Restore the original registry '
                         'or regenerate predictions in a separate evaluation cache; do not adopt a new hash.')
    destination = prediction_destination(run)
    if not destination.is_file():
        raise FileNotFoundError(f'Registered test predictions are missing: {destination}')
    # Verify and load the SAME byte snapshot, avoiding a hash/load time-of-check gap.
    payload = destination.read_bytes()
    if hashlib.sha256(payload).hexdigest() != entry['sha256']:
        raise ValueError(f'Cached test predictions changed from their original hash: {destination}')
    with np.load(io.BytesIO(payload), allow_pickle=False) as saved:
        arrays = {name:saved[name].copy() for name in saved.files}
    for field, expected in test_prediction_provenance(run).items():
        if field not in arrays or arrays[field].shape != () or arrays[field].item() != expected:
            raise ValueError(f'Cached test prediction provenance differs ({field})')
    required = {'logits', 'image_ids', 'labels', 'locations', 'seq_ids', 'class_names'}
    if not required.issubset(arrays):
        raise ValueError('Cached test predictions are missing required arrays')
    check_arrays(arrays, frame)
    return arrays


def record_generated_test_predictions(run, state, original_hash):
    # Only the fresh-inference branch may call this, with the hash of its newly written .tmp file.
    verify_test_export_registry_unchanged(state)
    key = (run['model'], int(run['seed']))
    if state['sealed'] or key in state['entries']:
        raise ValueError('An original prediction hash is already recorded; it cannot be replaced')
    destination = prediction_destination(run)
    if not destination.is_file() or sha(destination) != original_hash:
        raise ValueError('New test-prediction bytes changed before their creation hash was recorded')
    entry = {'model':key[0], 'seed':key[1], 'file':str(destination.relative_to(ROOT)), 'sha256':original_hash}
    updated = {**state['record'], 'exports':state['record']['exports']+[entry]}
    entries = validate_test_export_record(updated)
    # Journal each completed inference immediately, so a partial six-run pass can resume safely.
    write(state['path'], jtext(updated))
    state.update(record=updated, entries=entries, fingerprint=sha(state['path']))


registry = load_test_export_registry()
TEST = {}
for run in RUNS:
    key = (run['model'], int(run['seed']))
    destination = prediction_destination(run)
    provenance = test_prediction_provenance(run)
    started = time.perf_counter()
    if key in registry['entries'] or destination.exists():
        a = load_cached_test_predictions(run, registry, test_frame)
        print('Reusing hash-verified test logits:', *key, flush=True)
    else:
        print('Test inference:', *key, flush=True)
        checkpoint = load_verified_checkpoint(run)
        model = build_model(key[0]).to(DEVICE)
        model.load_state_dict(checkpoint['state_dict'], strict=True)
        model.eval()
        parts, ids = [], []
        loader = DataLoader(TestDataset(), batch_size=CFG['test_batch_size'], shuffle=False, num_workers=0)
        with torch.inference_mode():
            for batch, (x, image_ids) in enumerate(loader, 1):
                out = model(x.to(DEVICE)).cpu().numpy()
                if out.shape != (len(image_ids), K) or not np.isfinite(out).all():
                    raise ValueError('Test inference returned invalid shape or nonfinite logits')
                parts.append(out)
                ids.extend(image_ids)
                if batch % 100 == 0:
                    print(f'  {batch}/{len(loader)} batches', flush=True)
        a = {'logits':np.concatenate(parts), 'image_ids':np.asarray(ids, dtype=str),
             'labels':test_frame.label.to_numpy(dtype=np.int64), 'locations':test_frame.location.to_numpy(dtype=str),
             'seq_ids':test_frame.seq_id.to_numpy(dtype=str), 'class_names':np.asarray(CLASSES),
             **{k:np.asarray(v) for k,v in provenance.items()}}
        check_arrays(a, test_frame)
        tmp = Path(str(destination)+'.tmp')
        with tmp.open('wb') as output:
            np.savez_compressed(output, **a)
        original_hash = sha(tmp)
        tmp.replace(destination)
        record_generated_test_predictions(run, registry, original_hash)
        del model, checkpoint
        print(f'  Saved {len(ids)} predictions in {(time.perf_counter()-started)/60:.1f} minutes', flush=True)
    TEST[key] = a
verify_test_export_registry_unchanged(registry)
if set(registry['entries']) != {(r['model'], int(r['seed'])) for r in RUNS}:
    raise ValueError('The test-prediction registry is incomplete')
test_exports = registry['record']['exports']
# Existing original hashes are never replaced by hashes computed during cache reuse.


## 6. Test-set metrics and policy transfer

Temperature scaling preserves argmax, so raw and scaled classification accuracy and macro-F1 are identical **within each checkpoint**, while NLL, Brier and ECE may change. Test `accepted_error` is observed incorrect predictions **among accepted test images**, and `coverage` is the fraction accepted. If no policy threshold qualified on development cameras, **no test images are automatically accepted**: accepted error and `target_met_on_test` are undefined, not zero or false. The test column `target_met_on_test` is **descriptive**, not a post-test threshold tuning signal and not a confidence guarantee.

The v2 policy table contains development-split confidence bounds, whereas final test-camera bootstrap intervals estimate uncertainty of the already frozen decision. Keep those two kinds of uncertainty separate.

Seed summaries explicitly count contributing and abstaining seeds. Accepted-error means are
labelled as conditional on seeds with test acceptance; no-acceptance errors remain undefined.
`test_accepted_by_category.csv` separately groups accepted counts/errors by true category and
predicted category. Global classification precision/recall remain in `test_per_class.csv`.

A selected threshold can still accept no images on a particular test camera, or even no images
for an entire test model/seed/variant, because test confidences can differ from policy confidences.
With zero test acceptance, coverage is zero and accepted error is undefined, even if a threshold
was selected. Some accepted predictions with no mistakes instead give observed accepted error zero.

Policy counts/bounds describe the policy partition; `test_results.csv` describes transfer of the
frozen threshold to the separate test partition. A policy bound below 5% does not force test
error below 5%, and a selected zero-error policy does not predict zero test errors. True-category
acceptance tables show which reference categories are accepted or missed; predicted-category
tables show reliability of accepted labels emitted by the model. Neither changes the global
threshold or supplies category-specific confidence guarantees.


In [ ]:
def accepted_category_statistics(logits, y, accept):
    pred = logits.argmax(1)
    rows = []
    for grouping, values in [('true_category', y), ('predicted_category', pred)]:
        for c, name in enumerate(CLASSES):
            group = values == c
            accepted = group & accept
            support, n = int(group.sum()), int(accepted.sum())
            errors = int((accepted & (pred != y)).sum())
            rows.append({'grouping':grouping, 'category':name, 'support':support,
                         'accepted':n, 'accepted_errors':errors,
                         'coverage_within_group':n/support if support else np.nan,
                         'accepted_error':errors/n if n else np.nan})
    return rows


def summarize_seeds(result_table):
    rows = []
    for (model, variant), group in result_table.groupby(['model', 'variant'], sort=True):
        if group.seed.duplicated().any():
            raise ValueError('Repeated seed in result summary')
        contributors = group.accepted_error.notna() & group.accepted.gt(0)
        if not (group.accepted_error.notna() == group.accepted.gt(0)).all():
            raise ValueError('Accepted-error availability disagrees with accepted counts')
        row = {'model':model, 'variant':variant, 'seed_count':len(group),
               'accepted_error_contributing_seeds':int(contributors.sum()),
               'abstaining_seeds':int(group.accepted.eq(0).sum()),
               'seeds_with_selected_policy':int(group.policy_status.eq('selected').sum()),
               'accepted_error_mean_among_accepting_seeds':float(group.loc[contributors, 'accepted_error'].mean()),
               'accepted_error_std_among_accepting_seeds':float(group.loc[contributors, 'accepted_error'].std())}
        for metric in ['accuracy', 'macro_f1', 'nll', 'brier', 'ece', 'coverage']:
            row[metric+'_mean'] = float(group[metric].mean())
            row[metric+'_std'] = float(group[metric].std())
        rows.append(row)
    return pd.DataFrame(rows)


def selective(logits,y,locations,T,policy):
    conf=probabilities(logits,T)[0].max(1)
    accept=np.zeros(len(y),dtype=bool) if policy["threshold"] is None else conf>=policy["threshold"]
    wrong=logits.argmax(1)!=y;n=int(accept.sum());err=int(wrong[accept].sum())
    return {"accepted":n,"accepted_errors":err,"coverage":n/len(y),"manual_review_fraction":1-n/len(y),
        "accepted_error":err/n if n else np.nan,
        "accepted_cameras":len(np.unique(locations[accept])),
        "target_met_on_test":bool(err/n<=CFG["target_error"]) if n else None},accept

result_rows=[];class_rows=[];prediction_rows=[];ece_rows=[];accepted_category_rows=[]
for key,a in TEST.items():
    z,y,loc=a["logits"],a["labels"],a["locations"];pred=z.argmax(1)
    cm=np.bincount(y*K+pred,minlength=K*K).reshape(K,K)
    for variant,T in [("raw",1.),("scaled",TEMPERATURES[key])]:
        score=metrics(z,y,T);policy=POLICIES[key][variant]
        sel,accept=selective(z,y,loc,T,policy)
        result_rows.append({"model":key[0],"seed":key[1],"variant":variant,"temperature":T,
            "images":len(y),"cameras":len(np.unique(loc)),"threshold":policy["threshold"],
            "policy_status":policy["status"],"policy_accepted_error":policy["accepted_error"],
            "policy_screening_upper":policy["screening_upper"],**score,**sel})
        p,_=probabilities(z,T)
        for row in accepted_category_statistics(z, y, accept):
            accepted_category_rows.append({'model':key[0], 'seed':key[1], 'variant':variant, **row})
        for c,name in enumerate(CLASSES):
            tp=cm[c,c];support=cm[c].sum();predicted=cm[:,c].sum()
            group=(y==c)&accept;n=int(group.sum())
            class_rows.append({"model":key[0],"seed":key[1],"variant":variant,"class":name,
                "support":int(support),"predicted_count":int(predicted),
                "precision":tp/predicted if predicted else np.nan,"recall":tp/support if support else np.nan,
                "f1":2*tp/(support+predicted) if support+predicted else 0.,
                "accepted":n,"accepted_errors":int(((pred!=y)&group).sum()),"coverage_within_true_class":n/support if support else np.nan,
                "accepted_error":float((pred[group]!=y[group]).mean()) if n else np.nan})
        for bins in [10,15,20]:
            rel=reliability(z,y,T,bins)
            ece_rows.append({"model":key[0],"seed":key[1],"variant":variant,"bins":bins,
                "ece":float((rel["count"]*(rel.confidence-rel.accuracy).abs()).sum()/len(y))})
        prediction_rows.append(pd.DataFrame({"model":key[0],"seed":key[1],"variant":variant,
            "image_id":a["image_ids"],"location":loc,"true_class":[CLASSES[i] for i in y],
            "predicted_class":[CLASSES[i] for i in pred],"confidence":p.max(1),"correct":pred==y,"accepted":accept}))
results=table("test_results.csv",result_rows)
for key in TEST:
    r=results.loc[(results.model==key[0])&(results.seed==key[1])]
    if r.accuracy.nunique() != 1 or r.macro_f1.nunique() != 1:
        raise ValueError(f'Temperature scaling unexpectedly changed predicted-label metrics: {key}')
per_class=table("test_per_class.csv",class_rows)
table("test_accepted_by_category.csv",accepted_category_rows)
table("test_predictions.csv",pd.concat(prediction_rows,ignore_index=True))
table("ece_bin_sensitivity.csv",ece_rows)
summary = summarize_seeds(results)
write(REPORT/"test_seed_summary.csv", summary.to_csv(index=False))

display(results.round(4));display(summary.round(4))
print("Mean/std summarize training seeds, not independent camera-split replications.")

print("Accepted-error means are conditional on seeds with acceptance; contributing/abstaining counts are explicit.")


## 7. Descriptive paired whole-test-camera bootstrap

Resample the same **held-out test-camera IDs** for every model, seed and raw/scaled variant, retaining all events from each sampled camera. This respects within-camera dependence and preserves pairing. The 2,000 test bootstrap draws and their percentile 95% intervals are **descriptive**, condition on frozen training/calibration/policies, and do not constitute an additional threshold-selection step.

The test intervals do not include uncertainty from retraining, temperature refitting, new camera-split allocation, or selection of policy thresholds. With few cameras, their accuracy is limited. A reported policy-camera upper bound and a held-out test bootstrap interval describe different populations and computations; neither guarantees a future site's error rate.

The primary direct NLL-gain comparison uses the same paired camera draws and all three
seeds as the existing bootstrap. It is a difference of improvements, not a comparison of
separate significance decisions. Intervals condition on the fitted models, temperatures and policies.

Keep these three calculations distinct:

| Calculation | Data used | Purpose |
| --- | --- | --- |
| One-sided Bonferroni-adjusted binomial upper bound | Policy events | Select under independent-event assumptions |
| Approximate simultaneous camera-bootstrap upper estimate; 10,000 draws | Whole policy cameras | Diagnose camera sensitivity; never veto selection |
| Two-sided percentile camera-bootstrap interval; 2,000 shared draws | Whole test cameras | Describe uncertainty of fixed metrics and paired model/gain comparisons |

The test endpoints are the 2.5th and 97.5th percentiles of finite replicates; valid/total
replicate counts are reported. A test resample with no accepted predictions has undefined
accepted error, not zero error. If all observed accepted test predictions are correct, a
resampling interval may collapse to $[0,0]$. That reflects the observed sample's lack of errors,
not proof of zero future error or a formal camera-aware upper bound. Such intervals must not
replace the policy binomial calculation or trigger threshold changes. Few cameras and
concentrated errors can make finite bootstrap estimates unstable. More bootstrap repetitions
reduce simulation noise, not the shortage of independent cameras.


In [ ]:
cameras=np.unique(test_frame.location.to_numpy(dtype=str));C=len(cameras)
if C<2:raise ValueError("Camera bootstrap needs multiple test cameras")
cam_idx=np.searchsorted(cameras,test_frame.location.to_numpy(dtype=str))
rng=np.random.default_rng(CFG["bootstrap_seed"])
# Each row contains camera multiplicities for one shared bootstrap draw.
W=rng.multinomial(C,np.full(C,1/C),size=CFG["bootstrap_repeats"]).astype(float)
METRICS=["accuracy","nll","brier","ece","macro_f1","coverage","accepted_error"]
BOOT={}
def bootstrap_metrics(a,T,accept):
    z,y=a["logits"],a["labels"];p,lp=probabilities(z,T);pred=z.argmax(1);correct=pred==y
    conf=p.max(1);bins=CFG["ece_bins"];bi=np.minimum((conf*bins).astype(int),bins-1)
    # Column blocks: n, correct, NLL, Brier, accepted, accepted wrong, confusion, bin count/confidence/correct.
    values=np.zeros((len(y),6+K*K+3*bins))
    values[:,:6]=np.column_stack([np.ones(len(y)),correct,-lp[np.arange(len(y)),y],
        ((p-np.eye(K)[y])**2).sum(1),accept,accept&~correct])
    values[np.arange(len(y)),6+y*K+pred]=1
    start=6+K*K
    values[np.arange(len(y)),start+bi]=1
    values[np.arange(len(y)),start+bins+bi]=conf
    values[np.arange(len(y)),start+2*bins+bi]=correct
    agg=np.zeros((C,values.shape[1]));np.add.at(agg,cam_idx,values)
    s=W@agg;n=s[:,0];cm=s[:,6:6+K*K].reshape(-1,K,K)
    den=cm.sum(1)+cm.sum(2)
    f1=np.divide(2*np.diagonal(cm,axis1=1,axis2=2),den,out=np.zeros_like(den),where=den>0).mean(1)
    # count * |confidence mean - correct mean| = |confidence sum - correct sum|.
    ece=np.abs(s[:,start+bins:start+2*bins]-s[:,start+2*bins:start+3*bins]).sum(1)/n
    err=np.divide(s[:,5],s[:,4],out=np.full(len(n),np.nan),where=s[:,4]>0)
    return dict(zip(METRICS,[s[:,1]/n,s[:,2]/n,s[:,3]/n,ece,f1,s[:,4]/n,err]))

def interval(values):
    finite=values[np.isfinite(values)]
    return (float(np.quantile(finite,.025)),float(np.quantile(finite,.975)),len(finite)) if len(finite) else (np.nan,np.nan,0)
ci_rows=[]
for key,a in TEST.items():
    for variant,T in [("raw",1.),("scaled",TEMPERATURES[key])]:
        _,accept=selective(a["logits"],a["labels"],a["locations"],T,POLICIES[key][variant])
        samples=bootstrap_metrics(a,T,accept);BOOT[(*key,variant)]=samples
        point=results.loc[(results.model==key[0])&(results.seed==key[1])&(results.variant==variant)].iloc[0]
        for metric,v in samples.items():
            lo,hi,n=interval(v)
            ci_rows.append({"model":key[0],"seed":key[1],"variant":variant,"metric":metric,
                "estimate":point[metric],"ci_low":lo,"ci_high":hi,"valid_replicates":n,"total_replicates":len(W)})
    print("Bootstrapped:",*key,flush=True)
table("test_camera_bootstrap_intervals.csv",ci_rows)
contrast_rows=[]
def record_contrast(label,metric,values,estimate):
    lo,hi,n=interval(values)
    contrast_rows.append({"comparison":label,"metric":metric,"estimate":estimate,
        "ci_low":lo,"ci_high":hi,"valid_replicates":n,"total_replicates":len(W)})
def strict_mean(arrays):return np.mean(np.stack(arrays),axis=0)  # Undefined replicates remain undefined.
for model_name in train_cfg["models"]:
    for metric in METRICS:
        deltas=[BOOT[(model_name,s,"scaled")][metric]-BOOT[(model_name,s,"raw")][metric] for s in train_cfg["planned_seeds"]]
        r=results.loc[results.model.eq(model_name)]
        points=r.pivot(index="seed",columns="variant",values=metric)
        point=float((points.scaled-points.raw).mean()) if points.notna().all().all() else np.nan
        record_contrast(f"{model_name}: scaled minus raw (mean seeds)",metric,strict_mean(deltas),point)
for variant in ["raw","scaled"]:
    for metric in METRICS:
        deltas=[BOOT[("efficientnet_b0",s,variant)][metric]-BOOT[("mobilenet_v3_large",s,variant)][metric] for s in train_cfg["planned_seeds"]]
        points=results.loc[results.variant.eq(variant)].pivot(index="seed",columns="model",values=metric)
        point=float((points.efficientnet_b0-points.mobilenet_v3_large).mean()) if points.notna().all().all() else np.nan
        record_contrast(f"{variant}: EfficientNet minus MobileNet (mean seeds)",metric,strict_mean(deltas),point)
def calibration_gain_comparisons(boot, result_table, seeds):
    if list(seeds) != [42, 123, 2026]:
        raise ValueError('The planned primary comparison requires all three paired seeds')
    gains = {}
    points = {}
    rows = []
    for model in ['mobilenet_v3_large', 'efficientnet_b0']:
        part = result_table.loc[result_table.model.eq(model)]
        pair = part.pivot(index='seed', columns='variant', values='nll').reindex(seeds)
        if pair[['raw', 'scaled']].isna().any().any():
            raise ValueError('Missing paired NLL scores in the primary calibration-gain comparison')
        points[model] = float((pair.raw-pair.scaled).mean())
        gains[model] = strict_mean([boot[(model, seed, 'raw')]['nll']-
                                  boot[(model, seed, 'scaled')]['nll'] for seed in seeds])
        lo, hi, valid = interval(gains[model])
        rows.append({'comparison':model+' NLL gain (raw minus scaled; mean seeds)', 'metric':'nll',
                     'primary_metric':True, 'estimate':points[model], 'ci_low':lo, 'ci_high':hi,
                     'valid_replicates':valid, 'total_replicates':len(gains[model])})
    difference = gains['efficientnet_b0']-gains['mobilenet_v3_large']
    lo, hi, valid = interval(difference)
    rows.append({'comparison':'EfficientNet NLL gain minus MobileNet NLL gain (mean paired seeds)',
                 'metric':'nll', 'primary_metric':True,
                 'estimate':points['efficientnet_b0']-points['mobilenet_v3_large'],
                 'ci_low':lo, 'ci_high':hi, 'valid_replicates':valid, 'total_replicates':len(difference)})
    return pd.DataFrame(rows)


calibration_gains = table('calibration_gain_comparison.csv',
    calibration_gain_comparisons(BOOT, results, train_cfg['planned_seeds']))
contrast_rows.extend(calibration_gains.to_dict(orient='records'))
contrasts=table("paired_camera_bootstrap_contrasts.csv",contrast_rows)
display(contrasts.round(4))

display(calibration_gains.round(5))


## 8. Plots for interpretation
Reliability: the diagonal is perfect top-label calibration; below it indicates overconfidence within that bin. Small bins are noisy; marker size represents count. Each panel is one fixed trained run.

Risk–coverage: sort by descending confidence, include complete tied-score groups, and plot error among accepted predictions versus fraction accepted. These curves are **descriptive test diagnostics**, not permission to select a new test threshold. The horizontal line is the 5% target. Confusion matrices use all test inputs and are identical before/after temperature scaling.


In [ ]:
fig,axes=plt.subplots(2,3,figsize=(13,8))
rel_tables=[]
for ax,(key,a) in zip(axes.flat,TEST.items()):
    ax.plot([0,1],[0,1],"k--",lw=1)
    for variant,T in [("raw",1.),("scaled",TEMPERATURES[key])]:
        rel=reliability(a["logits"],a["labels"],T,CFG["ece_bins"])
        rel_tables.append(rel.assign(model=key[0],seed=key[1],variant=variant))
        good=rel["count"]>0
        ax.plot(rel.loc[good,"confidence"],rel.loc[good,"accuracy"],label=variant,alpha=.8)
        ax.scatter(rel.loc[good,"confidence"],rel.loc[good,"accuracy"],s=10+rel.loc[good,"count"]*.15)
    ax.set(title=f"{key[0]} | seed {key[1]}",xlabel="Mean confidence",ylabel="Accuracy",xlim=(0,1),ylim=(0,1))
    ax.legend()
fig.tight_layout();fig.savefig(REPORT/"test_reliability.png",dpi=160);plt.show();plt.close(fig)
table("test_reliability_bins.csv",pd.concat(rel_tables,ignore_index=True))

fig,axes=plt.subplots(2,3,figsize=(13,8));risk_tables=[]
for ax,(key,a) in zip(axes.flat,TEST.items()):
    for variant,T in [("raw",1.),("scaled",TEMPERATURES[key])]:
        conf=probabilities(a["logits"],T)[0].max(1);order=np.argsort(-conf,kind="stable")
        ends=np.r_[np.flatnonzero(conf[order][:-1]!=conf[order][1:]),len(order)-1]
        risk=np.cumsum(a["logits"].argmax(1)[order]!=a["labels"][order])[ends]/(ends+1)
        cov=(ends+1)/len(order)
        ax.plot(cov,risk,label=variant)
        risk_tables.append(pd.DataFrame({"model":key[0],"seed":key[1],"variant":variant,"coverage":cov,"error":risk}))
    ax.axhline(CFG["target_error"],color="gray",ls="--",lw=1)
    ax.set(title=f"{key[0]} | seed {key[1]}",xlabel="Coverage",ylabel="Accepted error",xlim=(0,1),ylim=(0,1));ax.legend()
fig.tight_layout();fig.savefig(REPORT/"test_risk_coverage.png",dpi=160);plt.show();plt.close(fig)
table("test_risk_coverage.csv",pd.concat(risk_tables,ignore_index=True))

fig,axes=plt.subplots(2,3,figsize=(14,9));cm_rows=[]
for ax,(key,a) in zip(axes.flat,TEST.items()):
    cm=np.bincount(a["labels"]*K+a["logits"].argmax(1),minlength=K*K).reshape(K,K)
    row_sum=cm.sum(1,keepdims=True)
    normalized=np.divide(cm,row_sum,out=np.zeros_like(cm,dtype=float),where=row_sum>0)
    ax.imshow(normalized,vmin=0,vmax=1,cmap="Blues")
    for i in range(K):
        for j in range(K):
            ax.text(j,i,str(cm[i,j]),ha="center",va="center",color="white" if normalized[i,j]>.5 else "black")
            cm_rows.append({"model":key[0],"seed":key[1],"true_class":CLASSES[i],"predicted_class":CLASSES[j],"count":int(cm[i,j])})
    ax.set(xticks=range(K),yticks=range(K),xticklabels=CLASSES,yticklabels=CLASSES,
           title=f"{key[0]} | seed {key[1]}",xlabel="Predicted",ylabel="True")
    ax.tick_params(axis="x",rotation=45)
fig.tight_layout();fig.savefig(REPORT/"test_confusion_matrices.png",dpi=160);plt.show();plt.close(fig)
table("test_confusion_matrices.csv",cm_rows)


## 9. Write the v2 methods, statistical caveats and completion record

All generated results must use the actual v2 manifest, counts, trained outputs and selected policies; do not carry forward v1 dataset counts, checkpoint scores, policy success claims or report results. The handoff includes the *frozen candidate grid*, the multiplicity correction, the camera-bootstrap method, and the exact distinction between **binomial policy screening with camera diagnostics** and **test evaluation**. Save the executed notebook so its outputs are preserved.


In [ ]:
def plain_table(df):
    return '```text\n' + df.to_string(index=False,float_format=lambda x:f'{x:.4f}') + '\n```'

n_selected = sum(1 for row in policy_rows if row['status']=='selected')
notes = f"""# Calibration and uncertainty-aware policy evaluation — v2 actual results
Generated {now()}. Evaluation hash: {EVAL_SHA}. Frozen decision hash: {DECISION_SHA}.
Experiment: {EXPERIMENT_ID}. Training ID: {TRAIN_ID}. Evaluated all {len(RUNS)} planned checkpoints.

## Calibration (separate camera partition)
Each corrected Notebook 03 v2 selected head-or-fine-tuning checkpoint is kept unchanged. One positive scalar temperature per run fitted by minimizing **image-weighted calibration NLL** within {CFG['temperature_bounds']}. Calibration fitting does not alter the CNN or argmax labels. The calibrated probability quality is evaluated independently on the held-out test split.

## Binomial-bound threshold selection (secondary analysis; policy cameras only)
For each raw and scaled model/seed output, compute confidence as the maximum softmax probability. Accept if confidence >= threshold and abstain on the rest; no manual image review is performed in this experiment. We fixed {N_THRESHOLDS} cutoffs in the interval [{THRESHOLD_GRID[0]}, {THRESHOLD_GRID[-1]}] before reading policy outcomes, including a finer high-confidence tail. Candidates are not extracted from policy observations. Ties are accepted together.

For each candidate, observe n accepted policy images and e wrong predictions, and compute a one-sided exact Clopper–Pearson 95%-familywise *binomial* upper bound on its unknown accepted error. Bonferroni uses alpha = {CFG['policy_familywise_alpha']}/({CFG['policy_num_policies']} policies x {N_THRESHOLDS} cutoffs) = {ALPHA_PER_CANDIDATE:.9g} per test. Binomial exactness assumes independent observations at a fixed cutoff. The multiple-testing adjustment allows threshold selection among all fixed cutoffs under that independence model.

For camera diagnostics only, we also ran {CFG['policy_camera_bootstrap_repeats']} seeded whole-policy-camera resamples over {N_POLICY_CAMERAS} distinct policy cameras, shared across all policies. We computed an approximate standardized-bootstrap critical value {POLICY_CRITICAL_TEXT} from the largest standardized positive bootstrap deviation across eligible cutoffs and policies, then formed per-cutoff approximate simultaneous camera-bootstrap upper estimates only where nondegenerate evidence exists. Zero observed errors, zero/near-zero variation and nonfinite estimates imply an unavailable camera diagnostic, not zero camera uncertainty and not rejection of the threshold. All images in a sampled camera stay together, and error/coverage remain image-weighted within the sampled mixture. Bootstrap replicates with no accepted images retain the original 100%-risk assignment in this diagnostic. Neither camera-estimate availability nor its value is used in selection.

The selection criterion uses only **binomial_bonferroni_ucb_95 <= {CFG['target_error']:.0%}**, with >= {CFG['minimum_accepted']} accepted images and >= {CFG['minimum_accepted_cameras']} contributing cameras. Zero-error candidates are allowed when this bound passes. For schema compatibility, screening_upper is an alias of the binomial bound. Camera-bootstrap estimates remain separate diagnostics and never veto a candidate. Among passing candidates, choose the greatest number of accepted images; tie-break by the higher cutoff. No passing cutoff means no automatic predictions (undefined accepted error), not zero-error success. {n_selected}/{len(policy_rows)} model/seed/variant policies selected a cutoff on the v2 policy data. Each candidate's observed risk and both upper bounds are available in its policy_candidates CSV. Raw and scaled policies were selected separately. All selections and temperatures were frozen before inference on test images.

**Interpretation of uncertainty:** The Clopper–Pearson bound is exact for independent accepted images and fixed cutoffs (with Bonferroni protecting the fixed-grid search), but the events are clustered within cameras. The camera-bootstrap max-t upper bound acknowledges camera structure, but with a modest number of cameras it is *approximate*, not an exact finite-sample interval. Selection by the binomial bound is assumption-dependent, **not a mathematical guarantee of <=5% errors at a newly deployed camera**. A missing camera diagnostic remains explicitly missing even when a threshold is selected; a finite diagnostic is not proof of valid camera-aware coverage. All risk estimates describe the available selected event/camera distributions; shifts in camera mix, animal class prevalence and unseen locations can defeat transfer. A policy with no threshold passing this screening rule may indicate insufficient evidence rather than a poor classifier.

## Temperatures (fitted development split)
{plain_table(temperature_summary)}

## Threshold decisions (policy split, with upper confidence bounds)
{plain_table(policy_summary)}

## Held-out test evaluation
{len(test_frame)} distinct event representatives from {C} previously excluded classifier camera locations. The five categories and camera-specific sample sizes come from the frozen v2 data selection. Accuracy, five-class macro-F1, NLL, summed multiclass Brier, 15-bin top-label ECE (10/20-bin sensitivity), coverage and accepted-error are reported without test-driven changes. If a policy accepts no images, accepted error and empirical test-target status remain undefined.

{plain_table(results)}

## Seed summaries and category-conditioned acceptance
Accepted-error summaries are means among seeds with acceptance, with the contributing seed count,
abstaining seed count, and selected-policy seed count shown explicitly. Undefined errors remain missing.
{plain_table(summary)}
Accepted counts/errors, support, coverage and error rate are reported separately by true and predicted
category in test_accepted_by_category.csv. The full camera audit is deferred to Notebook 05 v2.
Manual crop verification was not performed in this experiment; automatic integrity checks do not
establish semantic crop-label agreement.

## Primary paired NLL calibration-gain comparison
Gain is raw minus scaled NLL. The architecture comparison is EfficientNet gain minus MobileNet gain,
averaged across all three paired seeds in each shared whole-test-camera resample. NLL is primary;
this comparison was fixed before test inference. Positive values favour a larger EfficientNet gain,
negative values a larger MobileNet gain. A zero-crossing interval is inconclusive, not equivalence.
{plain_table(calibration_gains)}

## Held-out test uncertainty
{CFG['bootstrap_repeats']} paired whole-test-camera bootstrap draws with seed {CFG['bootstrap_seed']} form descriptive two-sided percentile 95% intervals and architecture/temperature contrasts. Same test cameras are used in all model/seed replicates. The estimates condition on model weights, calibration temperatures and selected thresholds; they do not include retraining, split selection, or temperature/threshold refitting uncertainty. Policy-camera upper bounds are NOT the test intervals. Training seed means/std reflect a fixed camera split.

## Scope and limitations
Classifier training and selection follow the v2 one-frame-per-eligible-event, camera-stratified, development-camera-capped pipeline. The camera counts and class distributions were not copied from v1. Notebook 02 v2 uses official precomputed MDv5a animal boxes and full-frame fallback, so species-label or crop semantic correctness is not guaranteed. Detector training may have included Caltech images; camera holdout refers to downstream classifier training, not detector pretraining. Rare categories, differing camera environments and the cropped/full-frame mixture can affect performance. Training head/fine-tuning choices and checkpoint selection remain as frozen in Notebook 03 v2. Results cannot isolate architecture from its particular pretrained weights. No claim of verified future error control is made.

## Outputs and references
Tables: temperature_summary.csv, policy_summary.csv, twelve policy_candidates_*.csv, test_results.csv, test_seed_summary.csv, test_per_class.csv, test_accepted_by_category.csv, calibration_gain_comparison.csv, ece_bin_sensitivity.csv, test_camera_bootstrap_intervals.csv, paired_camera_bootstrap_contrasts.csv. Figures: test_reliability.png, test_risk_coverage.png, test_confusion_matrices.png. Audit files: evaluation_protocol.json, frozen_calibration_and_policies.json, test_exports.json. Each test-prediction creation hash is journalled immediately and checked before cache reuse; existing hashes are never re-baselined. Completed evaluations anchor the registry hash in completion.json. Input/provenance checks use explicit exceptions, not removable assertions. Model/logit files are under models/{EXPERIMENT_ID}/evaluation_v2 and the {TRAIN_ID} directories.

Guo et al. (2017), On Calibration of Modern Neural Networks: https://proceedings.mlr.press/v70/guo17a.html
Clopper & Pearson (1934), The Use of Confidence or Fiducial Limits Illustrated in the Case of the Binomial: https://doi.org/10.1093/biomet/26.4.404
Hoeffding/Bonferroni style multiple-threshold selection and limitations: describe the fixed grid and statistical assumptions above; no external selective-risk guarantee is asserted for this clustered setup.
PyTorch weights-only loading: https://docs.pytorch.org/docs/stable/generated/torch.load.html
MegaDetector results: https://lila.science/megadetector-results-for-camera-trap-datasets/
"""
write(REPORT / 'FINAL_METHODS_AND_RESULTS.md', notes)
write(REPORT / 'completion.json', jtext({
    'time_utc':now(),'experiment_id':EXPERIMENT_ID,
    'evaluation_sha256':EVAL_SHA,'decisions_sha256':DECISION_SHA,
    'completed_runs':len(TEST),'test_images':len(test_frame),'test_cameras':C,
    'policy_cameras':N_POLICY_CAMERAS,
    'policy_bound_type':'independent-event binomial Clopper-Pearson Bonferroni; camera bootstrap diagnostic only',
    'policy_selection_bound':CFG['policy_selection_bound'],
    'camera_uncertainty_role':CFG['camera_uncertainty_role'],
    'policy_bound_guaranteed_under_camera_shift':False,
    'camera_aware_95pct_guarantee':False, 'primary_calibration_gain_metric':'nll',
    'policies_with_threshold':n_selected,
    'bootstrap_repeats':CFG['bootstrap_repeats'],
    'test_used_for_fitting':False,
    'outputs':{p.name:sha(p) for p in sorted(REPORT.iterdir())
               if p.is_file() and p.name != 'completion.json'},
}))
print('Completed v2 evaluation on all six runs.')
print('Frozen policy UCB rule:', CFG['policy_method'])
print(f'Policy selections: {n_selected}/{len(policy_rows)} raw/scaled model-seed cases')
print('Report tables, plots and methods:',REPORT)
print('Save this executed notebook with Ctrl+S.')
